In [1]:
import json
import time
notebook_start = time.perf_counter()

import os, json, pandas as pd, numpy as np, matplotlib.pyplot as plt, shap, joblib, thermoift.PLOT_SETTINGS as ps
from matplotlib.lines import Line2D
from matplotlib.ticker import AutoMinorLocator, LogLocator
from sklearn.model_selection import train_test_split, cross_validate, KFold
from xgboost import XGBRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from thermoift.rng_utils import get_rng
from thermoift import MLPostprocessing, plot_correlation_heatmap, print_model_metrics

# Respect SLURM CPU allocation — prevents all 254 node CPUs being grabbed
n_cpus = int(os.environ.get("SLURM_CPUS_PER_TASK", 4))
os.environ["OMP_NUM_THREADS"]      = str(n_cpus)
os.environ["MKL_NUM_THREADS"]      = str(n_cpus)
os.environ["OPENBLAS_NUM_THREADS"] = str(n_cpus)
os.environ["NUMEXPR_NUM_THREADS"]  = str(n_cpus)
print(f"Thread limit set to {n_cpus} (SLURM_CPUS_PER_TASK)")

Thread limit set to 4 (SLURM_CPUS_PER_TASK)


In [2]:
OUTPUT_FOLDER = "XGBBUBBLE_OUTPUTS"
SEED          = 455552
TEST_ROWS     = None
DATA_PATH     = ""

In [3]:
# Parameters
OUTPUT_FOLDER = "/scratch-shared/draju/PART_2/ACTIVELEARNING/XGB_OUTPUTS/AL_ST/N100/trial_09/XGBBubble"
TEST_ROWS = None
SEED = 50024
DATA_PATH = "/scratch-shared/draju/PART_2/ACTIVELEARNING/COMBINED/AL_ST/N100/trial_09.csv"


In [4]:
df = pd.read_csv(DATA_PATH)

if isinstance(TEST_ROWS, str) and TEST_ROWS.strip().lower() in ("", "none", "null"):
    TEST_ROWS = None
if TEST_ROWS is not None:
    TEST_ROWS = int(TEST_ROWS)
    df = df.iloc[:TEST_ROWS].copy()
    print(f"Test mode: using first {TEST_ROWS} rows only")

print(f"Total samples: {len(df)}")
print(f"\nP_bubble statistics:")
print(df["P_bubble"].describe())

Total samples: 19460

P_bubble statistics:
count    19460.000000
mean        54.867633
std         31.968789
min          2.606366
25%         30.282498
50%         51.960277
75%         73.990758
max        230.990154
Name: P_bubble, dtype: float64


In [5]:
target     = "P_bubble"
rng        = get_rng(seed=SEED)

z_columns  = [col for col in df.columns if col.startswith("z_")]
Z_non_zero = [col for col in z_columns if (df[col] != 0).any()]
features   = ["temperature", "pressure"] + Z_non_zero

print(f"Selected features: {features}")

X = df[features]
y = df[target]

# 70/15/15 split: first 70/30, then split 30 into 15/15
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=SEED)
X_test, X_val, y_test, y_val = train_test_split(X_temp, y_temp, test_size=0.50, random_state=SEED)

print(f"\nTraining samples:   {X_train.shape[0]}")
print(f"Testing samples:    {X_test.shape[0]}")
print(f"Validation samples: {X_val.shape[0]}")

Selected features: ['temperature', 'pressure', 'z_carbon dioxide', 'z_hydrogen', 'z_nitrogen', 'z_argon', 'z_methane', 'z_oxygen', 'z_carbon monoxide', 'z_hydrogen sulfide']

Training samples:   13622
Testing samples:    2919
Validation samples: 2919


In [6]:
# XgBoost Regressor
xgb_model = XGBRegressor(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=rng,
    n_jobs=n_cpus)

xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=False
)

y_train_pred = xgb_model.predict(X_train)
y_test_pred  = xgb_model.predict(X_test)
y_val_pred   = xgb_model.predict(X_val)

#Metrics    
metrics = print_model_metrics(y_train, y_train_pred, y_test, y_test_pred, target, y_val=y_val, y_val_pred=y_val_pred)

Model Performance for P_bubble

Training Set:
  R²:   0.999752
  RMSE: 0.503800 bar
  MAE:  0.384950 bar

Test Set:
  R²:   0.999537
  RMSE: 0.699844 bar
  MAE:  0.515810 bar

Validation Set:
  R²:   0.999496
  RMSE: 0.703871 bar
  MAE:  0.514338 bar


In [7]:
results_df = pd.DataFrame({
    "idx": np.concatenate([y_train.index, y_test.index, y_val.index]),
    "actual": np.concatenate([y_train.values, y_test.values, y_val.values]),
    "predicted": np.concatenate([y_train_pred, y_test_pred, y_val_pred]),
    "split": ["train"]*len(y_train) + ["test"]*len(y_test) + ["val"]*len(y_val),
})
os.makedirs(OUTPUT_FOLDER, exist_ok=True)
results_df.to_csv(os.path.join(OUTPUT_FOLDER, f"XGB_{target}_predictions.csv"), index=False)
print(f"Predictions saved: {len(results_df)} rows")

# Save fitted model for future use without retraining
model_path = os.path.join(OUTPUT_FOLDER, f"XGB_{target}_model.joblib")
joblib.dump(xgb_model, model_path)
print(f"Model saved to: {model_path}")

Predictions saved: 19460 rows
Model saved to: /scratch-shared/draju/PART_2/ACTIVELEARNING/XGB_OUTPUTS/AL_ST/N100/trial_09/XGBBubble/XGB_P_bubble_model.joblib


In [8]:
# 5-fold cross-validation — single pass, all 4 metrics (5 fits instead of 20)
# n_jobs=1: XGBRegressor already parallelizes internally; loky workers fight over /dev/shm
cv_results = cross_validate(
    xgb_model, X, y, cv=5,
    scoring={
        "r2":   "r2",
        "rmse": "neg_root_mean_squared_error",
        "mae":  "neg_mean_absolute_error",
        "mape": "neg_mean_absolute_percentage_error",
    },
    n_jobs=1,
)
cv_r2_scores   = cv_results["test_r2"]
cv_rmse_scores = -cv_results["test_rmse"]
cv_mae_scores  = -cv_results["test_mae"]
cv_mape_scores = -cv_results["test_mape"]

print(f"Cross-Validation R² Scores:   {cv_r2_scores}")
print(f"Mean CV R²:   {cv_r2_scores.mean():.6f} (+/- {cv_r2_scores.std() * 2:.6f})")
print(f"\nCross-Validation RMSE Scores: {cv_rmse_scores}")
print(f"Mean CV RMSE: {cv_rmse_scores.mean():.6f} (+/- {cv_rmse_scores.std() * 2:.6f})")
print(f"\nCross-Validation MAE Scores:  {cv_mae_scores}")
print(f"Mean CV MAE:  {cv_mae_scores.mean():.6f} (+/- {cv_mae_scores.std() * 2:.6f})")
print(f"\nCross-Validation MAPE Scores: {cv_mape_scores * 100}")
print(f"Mean CV MAPE: {cv_mape_scores.mean() * 100:.4f}% (+/- {cv_mape_scores.std() * 2 * 100:.4f}%)")

Cross-Validation R² Scores:   [0.98928398 0.97973053 0.97363976 0.98285552 0.9628992 ]
Mean CV R²:   0.977682 (+/- 0.017898)

Cross-Validation RMSE Scores: [4.02016981 4.37340656 5.57184706 3.41543512 4.69941599]
Mean CV RMSE: 4.416055 (+/- 1.435530)

Cross-Validation MAE Scores:  [2.75162918 3.13625992 3.57875477 2.50529606 3.07763433]
Mean CV MAE:  3.009915 (+/- 0.729646)

Cross-Validation MAPE Scores: [ 8.9926934   7.25063231  8.25366298  7.26890277 15.18507767]
Mean CV MAPE: 9.3902% (+/- 5.9402%)


In [9]:
metrics["cv_r2_scores"]   = cv_r2_scores.tolist()
metrics["cv_r2_mean"]     = float(cv_r2_scores.mean())
metrics["cv_r2_std"]      = float(cv_r2_scores.std())
metrics["cv_rmse_scores"] = cv_rmse_scores.tolist()
metrics["cv_rmse_mean"]   = float(cv_rmse_scores.mean())
metrics["cv_rmse_std"]    = float(cv_rmse_scores.std())
metrics["cv_mae_scores"]  = cv_mae_scores.tolist()
metrics["cv_mae_mean"]    = float(cv_mae_scores.mean())
metrics["cv_mae_std"]     = float(cv_mae_scores.std())
metrics["cv_mape_scores"] = (cv_mape_scores * 100).tolist()
metrics["cv_mape_mean"]   = float(cv_mape_scores.mean() * 100)
metrics["cv_mape_std"]    = float(cv_mape_scores.std() * 100)
metrics["model"]          = "XGBoost"
metrics["features"]       = list(features)
metrics["target"]         = target
metrics["seed"]           = SEED

os.makedirs(OUTPUT_FOLDER, exist_ok=True)
metrics_path = os.path.join(OUTPUT_FOLDER, f"XGB_{target}_metrics.json")
with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2, default=str)
print(f"Metrics saved to: {metrics_path}")

Metrics saved to: /scratch-shared/draju/PART_2/ACTIVELEARNING/XGB_OUTPUTS/AL_ST/N100/trial_09/XGBBubble/XGB_P_bubble_metrics.json


In [10]:
notebook_end = time.perf_counter()
elapsed_minutes = (notebook_end - notebook_start) / 60
print(f"Total notebook runtime: {elapsed_minutes:.2f} minutes")

Total notebook runtime: 0.10 minutes
